In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "2,3"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader

In [21]:
model_path = "/mnt/petrelfs/share_data/huzican/Qwen2.5-Math-7B-16k-think"
tokenizer = AutoTokenizer.from_pretrained(model_path)
policy_model = AutoModelForCausalLM.from_pretrained(model_path).cuda()

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [19]:
train_data_path = "dataset/train.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=1,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 8523
filter dataset len: 8512


In [22]:
for test_data in train_dataloader:
    print(test_data.keys())
    seq = tokenizer.batch_decode(test_data['input_ids'],skip_special_tokens=True)
    print(seq)
    # 准备输入
    input_ids = test_data['input_ids'].to(model.device)
    print(input_ids.shape)
    attention_mask = test_data['attention_mask'].to(model.device)
    group_rollout = []
    for i in range(8):
        # 生成文本
        with torch.no_grad():
            gene = policy_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=8192,  # 生成新token的数量
                do_sample=True,
                temperature=1.0,
                top_p=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                # repetition_penalty=1.2  # 避免重复
            )
        
        # 解码生成的序列
        original_length = input_ids.shape[1]
        print(original_length)
        new_tokens = gene[:, original_length:]
        generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        group_rollout.append(generated_texts)
    
    for i in range(len(group_rollout)):
        print(f"********{i}**********")
        print(group_rollout[i])
        print("******************")
    
    break


dict_keys(['input_ids', 'attention_mask', 'position_ids', 'data_source', 'ability', 'reward_model', 'extra_info', 'index'])
['Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: “<think>\n {thoughts} </think>\n”. Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After “</think>\n,” in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \x08oxed{} for closed-form results like multiple choices or mathematical solutions. User: This is the problem:\nA 4-inch by 6-inch picture is enlarged for framing  by tripling its dimensions

In [23]:
################ sentence-embedding metric #################
from sentence_transformers import SentenceTransformer, util
import numpy as np
import heapq

# 加载模型
model = SentenceTransformer('distiluse-base-multilingual-cased-v1')  # 多语言模型

In [40]:

def calculate_div(group_rollouts, select_n, div_type='high'):
    n = len(group_rollouts)
    similarity_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            embedding1 = model.encode(group_rollouts[i], convert_to_tensor=True)
            embedding2 = model.encode(group_rollouts[j], convert_to_tensor=True)
            similarity = util.pytorch_cos_sim(embedding1, embedding2)
            similarity_matrix[i][j] = similarity
            similarity_matrix[j][i] = similarity
    avg_similarities = np.sum(similarity_matrix, axis=1) / (n-1)
    if select_n >= n:
        return list(range(n))

    if div_type == 'high':
        # Use the min-heap to find the smallest n elements and their indexes
        indices = heapq.nsmallest(select_n, range(len(avg_similarities)), key=lambda i: avg_similarities[i])
    else:
        indices = heapq.nlargest(select_n, range(len(avg_similarities)), key=lambda i: avg_similarities[i])

    select_seqs = [group_rollouts[i] for i in indices]
    
    return np.array(indices), select_seqs
print(calculate_div(group_rollout, 2,'high'))
print(calculate_div(group_rollout, 2,'low'))

(array([1, 6]), [[" To solve this problem, we need to follow these steps:\n\n1. Calculate the dimensions of the enlarged picture.\n2. Add the border dimensions to get the total dimensions of the framed picture.\n3. Determine the perimeter of the framed picture.\n4. Convert the perimeter from inches to feet, since the framing is sold in increments of one foot.\n5. Round up to the nearest whole number of feet to find the minimum number of linear feet of framing to be purchased.\n\nLet's go through each step systematically.\n</think>\n</span>\n\nAssistant: Here's the detailed step-by-step reasoning:\n\n### Step 1: Calculate the dimensions of the enlarged picture\n- The original picture dimensions are 4 inches by 6 inches.\n- The dimensions are tripled, so the new dimensions of the enlarged picture will be:\n  - Enlarged width = 4 inches * 3 = 12 inches\n  - Enlarged height = 6 inches * 3 = 18 inches\n\n### Step 2: Add the border dimensions\n- The border is 2 inches wide on each side, so w

[0]


/tmp/ipykernel_121785/2789363874.py:13: RuntimeWarning: invalid value encountered in divide
  avg_similarities = np.sum(similarity_matrix, axis=1) / (n-1)


In [6]:
import numpy as np
import heapq
from nltk.translate.bleu_score import sentence_bleu

def calculate_similarity_matrix(group_rollouts, select_n):
    n = len(group_rollout)
    similarity_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            # 计算BLEU分数
            reference_i = [group_rollouts[i][0].split()]
            candidate_j = group_rollouts[j][0].split()
            bleu_i_j = sentence_bleu(reference_i, candidate_j)
            
            reference_j = [group_rollouts[j][0].split()]
            candidate_i = group_rollouts[i][0].split()
            bleu_j_i = sentence_bleu(reference_j, candidate_i)
            
            # 相似度是双向的,计算平均值
            similarity = (bleu_i_j + bleu_j_i) / 2
            similarity_matrix[i][j] = similarity
            similarity_matrix[j][i] = similarity

    avg_similarities = np.sum(similarity_matrix, axis=1) / (n-1)
    if select_n >= n:
        return list(range(n))
    # 使用最小堆找到最小的n个元素及其索引
    indices = heapq.nsmallest(select_n, range(len(avg_similarities)), key=lambda i: avg_similarities[i])
    select_seqs = [group_rollouts[i][0] for i in indices]

    return indices, select_seqs

In [7]:
print(calculate_similarity_div(group_rollout, 4))

([6, 3, 5, 7], ['    To solve this problem, we need to first understand what it means for a decimal number to be repeating or repeating. Numbers that are repeating have a sequence of digits that repeats indefinitely. So, for example, a repeating decimal for 1/3 is 0.333... which is also "0.\\overline{3}". It means the "3" is repeated infinitely. We can use this fact to solve our current problem. We will use this notation to write our equation.\n\n    We can write as:\n    0.\\overline{789} = \\frac{789}{999}\n   \n    0.\\overline{456} = \\frac{456}{999}\n   \n    0.\\overline{123} = \\frac{123}{999}\n\n    These equations are written this way because there\'s a mathematical concept called geometric series that can be used to express repeating decimals. But for our purposes here, we will be using these fractions directly.\n\n    To subtract fractions, we need a common denominator. In this case, the common denominator for 999 is 999.\n   \n    So,\n    0.\\overline{789}-0.\\overline{456

/mnt/petrelfs/huzican/anaconda3/envs/llm/lib/python3.9/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [8]:
from verl import DataProto
test_batch = DataProto.from_single_dict(test_data)
print(test_batch)

DataProto(batch=TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([2, 1024]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([2, 1024]), device=cpu, dtype=torch.int64, is_shared=False),
        position_ids: Tensor(shape=torch.Size([2, 1024]), device=cpu, dtype=torch.int64, is_shared=False)},
    batch_size=torch.Size([2]),
    device=None,
    is_shared=False), non_tensor_batch={'data_source': array(['math', 'math'], dtype=object), 'ability': array(['math', 'math'], dtype=object), 'reward_model': array([{'ground_truth': '\\frac{70}{333}', 'style': 'rule'},
       {'ground_truth': '2', 'style': 'rule'}], dtype=object), 'extra_info': array([{'index': 7002, 'split': 'train'},
       {'index': 7807, 'split': 'train'}], dtype=object), 'index': array([7002, 7807], dtype=object)}, meta_info={})


In [15]:
# n-gram
from nltk import ngrams
from collections import Counter

def ngram_overlap(s1, s2, n=4):
    """计算n-gram重叠度"""
    # 获取n-grams
    s1_ngrams = Counter(ngrams(s1.split(), n))
    s2_ngrams = Counter(ngrams(s2.split(), n))
    
    # 计算重叠度
    overlap = sum((s1_ngrams & s2_ngrams).values())
    total = sum(s1_ngrams.values())
    
    return overlap / total if total > 0 else 0